## Import Libraries

In [1]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

## Step 1. Data Acquisition

Fetch the dataset and load it into a DataFrame.

In [2]:
newsgroups = fetch_20newsgroups(subset='all',
                                  remove=('headers', 'footers', 'quotes'))

In [3]:
df = pd.DataFrame({'text': newsgroups.data, 'target': newsgroups.target})

In [4]:
df.head()

,text,target
0,\n\nI am sure some bashers of Pens fans are pr...,10
1,My brother is in the market for a high-perform...,3
2,\n\n\n\n\tFinally you said what you dream abou...,17
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,3
4,1) I have an old Jasmine drive which I cann...,4


In [5]:
print(newsgroups.target_names)

['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [6]:
for i, name in enumerate(newsgroups.target_names):
    print(i, "->", name)

0 -> alt.atheism
1 -> comp.graphics
2 -> comp.os.ms-windows.misc
3 -> comp.sys.ibm.pc.hardware
4 -> comp.sys.mac.hardware
5 -> comp.windows.x
6 -> misc.forsale
7 -> rec.autos
8 -> rec.motorcycles
9 -> rec.sport.baseball
10 -> rec.sport.hockey
11 -> sci.crypt
12 -> sci.electronics
13 -> sci.med
14 -> sci.space
15 -> soc.religion.christian
16 -> talk.politics.guns
17 -> talk.politics.mideast
18 -> talk.politics.misc
19 -> talk.religion.misc


## Step 2. Data Preprocessing

Clean and prepare the text data. This includes lowercasing, removing numbers,
 tokenization, punctuation, short words, stop words, and lemmatization.

In [7]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import string

In [8]:
# Lowercasing
df['text_processed'] = df['text'].apply(lambda doc: doc.lower())
df['text_processed'].head()

0    \n\ni am sure some bashers of pens fans are pr...
1    my brother is in the market for a high-perform...
2    \n\n\n\n\tfinally you said what you dream abou...
3    \nthink!\n\nit's the scsi card doing the dma t...
4    1)    i have an old jasmine drive which i cann...
Name: text_processed, dtype: str

In [9]:
# Cleaning - removing all numbers from text
df['text_processed'] = df['text_processed'].apply(lambda doc: re.sub(r'\d+', '', doc))
df['text_processed'].head()

0    \n\ni am sure some bashers of pens fans are pr...
1    my brother is in the market for a high-perform...
2    \n\n\n\n\tfinally you said what you dream abou...
3    \nthink!\n\nit's the scsi card doing the dma t...
4    )    i have an old jasmine drive which i canno...
Name: text_processed, dtype: str

In [10]:
# Remove punctuation
df['text_processed'] = df['text_processed'].apply(
    lambda doc: doc.translate(str.maketrans('', '', string.punctuation)))

In [11]:
# Tokenization the text by splitting it into words
df['text_processed'] = df['text_processed'].apply(lambda doc: doc.split())
df['text_processed'].head()

0    [i, am, sure, some, bashers, of, pens, fans, a...
1    [my, brother, is, in, the, market, for, a, hig...
2    [finally, you, said, what, you, dream, about, ...
3    [think, its, the, scsi, card, doing, the, dma,...
4    [i, have, an, old, jasmine, drive, which, i, c...
Name: text_processed, dtype: object

In [12]:
# Filtering out non-alphabetic tokens and short tokens
df['text_processed'] = df['text_processed'].apply(
    lambda tokens: [word for word in tokens if word.isalpha() and len(word) > 1])

In [13]:
# Stop-words removal
stop_words = set(stopwords.words('english'))

# Getting the set of English stopwords
df['text_processed'] = df['text_processed'].apply(lambda tokens: [w for w in tokens if w not in stop_words])
df['text_processed'].head()

0    [sure, bashers, pens, fans, pretty, confused, ...
1    [brother, market, highperformance, video, card...
2    [finally, said, dream, mediterranean, new, are...
3    [think, scsi, card, dma, transfers, disks, scs...
4    [old, jasmine, drive, cannot, use, new, system...
Name: text_processed, dtype: object

In [14]:
# Stemming
stemmer = PorterStemmer()
df['text_processed'] = df['text_processed'].apply(
    lambda tokens: [stemmer.stem(w) for w in tokens])

In [15]:
# examples of stemmer applied
words = ['running','runner','run']
for word in words:
    print(stemmer.stem(word))

run
runner
run


In [16]:
words = ['analyze', 'analysis', 'analyzed']
for word in words:
    print(stemmer.stem(word))


analyz
analysi
analyz


In [17]:
# Joining stemming results
df['text_processed'] = df['text_processed'].apply(' '.join)
df['text_processed'].head()

0    sure basher pen fan pretti confus lack kind po...
1    brother market highperform video card support ...
2    final said dream mediterranean new area greate...
3    think scsi card dma transfer disk scsi card dm...
4    old jasmin drive cannot use new system underst...
Name: text_processed, dtype: str

## Step 3. Feature Extraction

In this step, we will convert the cleaned text data into a numerical representation. We will apply two methods to see their effects on the final model: CountVectorizer and TfidfVectorizer. The main difference between them lies in the type of feature representation they produce. Finally, we will also split the dataset into training and testing sets, which is crucial for training and evaluating our machine learning model.

### 3.1 CountVectorizer Feature Extraction

In [18]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [19]:
# Train-test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(df['text_processed'],
                                                    df['target'], test_size=0.2, random_state=42)

In [20]:
# Display the sizes of training and test data

print("Step 3: Feature Extraction\n")
print("Size of training data:", len(X_train))
print("Size of testing data:", len(X_test))

Step 3: Feature Extraction

Size of training data: 15076
Size of testing data: 3770


In [21]:
# Vectorization using CountVectorization
# Fitting and transforming the training data into count format
# Transforming the test data into count format using the same fitted vectorizer

vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

### 3.2 TfidfVectorizer Feature Extraction

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [23]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [24]:
print("Shape of TF-IDF matrix for training data:", X_train_tfidf.shape)
print("Shape of TF-IDF matrix for testing data:", X_test_tfidf.shape, "\n")

Shape of TF-IDF matrix for training data: (15076, 81987)
Shape of TF-IDF matrix for testing data: (3770, 81987) 



## Step 4: Model Training and Evaluation
This step involves training a machine learning model to classify the text data. We will use the Multinomial Naive Bayes classifier, which is well-suited for text classification tasks and performs effectively with the TF-IDF features we've extracted.

We will also evaluate the performance of our trained models. We will make predictions on the test data and compute accuracy and a classification report that includes precision, recall, and F1-score for each class in the dataset. This evaluation is essential to understand how well the model performs on unseen data.


### 4.1  CountVectorizer Model Evaluation

In [25]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

In [26]:
model = MultinomialNB()

In [28]:
# Fit model on train set
model.fit(X_train_counts, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[648.,771.,790.,...,758.,616.,492.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.15,-2.97,-2.95,...,-2.99,-3.2 ,-3.42]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 81987)","[[20., 0., 0.,..., 0., 0., 0.], [ 0., 0., 0.,..., 0., 0., 1.], [ 1., 0., 0.,..., 0., 0., 0.], ..., [ 2., 0., 0.,..., 0., 0., 0.], [ 4., 1., 0.,..., 0., 0., 0.], [ 1., 0., 0.,..., 0., 0., 0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 81987)","[[ -8.83,-11.88,-11.88,...,-11.88,-11.88,-11.88], [-12.03,-12.03,-12.03,...,-12.03,-12.03,-11.34], [-11.16,-11.85,-11.85,...,-11.85,-11.85,-11.85], ..., [-11.2 ,-12.3 ,-12.3 ,...,-12.3 ,-12.3 ,-12.3 ], [-10.41,-11.33,-12.02,...,-12.02,-12.02,-12.02], [-11.1 ,-11.79,-11.79,...,-11.79,-11.79,-11.79]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,81987


In [29]:
# Make predictions on test data
y_pred = model.predict(X_test_counts)

In [30]:
print("Step 4: Model Evaluation\n")
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

Step 4: Model Evaluation

Classification Report:

              precision    recall  f1-score   support

           0       0.61      0.40      0.48       151
           1       0.52      0.73      0.61       202
           2       0.87      0.27      0.41       195
           3       0.53      0.74      0.62       183
           4       0.82      0.63      0.71       205
           5       0.74      0.80      0.77       215
           6       0.89      0.58      0.70       193
           7       0.86      0.67      0.75       196
           8       0.50      0.68      0.58       168
           9       0.92      0.78      0.85       211
          10       0.93      0.88      0.91       198
          11       0.64      0.78      0.70       201
          12       0.79      0.56      0.66       202
          13       0.83      0.80      0.82       194
          14       0.74      0.79      0.77       189
          15       0.46      0.91      0.61       202
          16       0.72      0.

## Check incorrect predictions

In [31]:
y_pred = pd.Series(index=y_test.index, data=y_pred)

In [32]:
pd.concat([y_test, y_pred], axis=1)

,target,0
18265,9,9
423,12,1
7900,14,14
14785,18,18
5217,0,14
...,...,...
16786,11,11
17397,3,1
9251,0,0
638,19,15
